# Customer Intelligence / Country Segmentation Notebook

## Unsupervised Learning on Country Data

This notebook implements a full clustering workflow using the Kaggle dataset **Unsupervised Learning on Country Data**.


## 1) Install required libraries

In [ ]:
!pip -q install pandas numpy matplotlib seaborn scikit-learn

## 2) Import libraries

In [ ]:
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

sns.set(style='whitegrid')
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

## 3) Load the dataset

Upload the CSV file from Kaggle when prompted.


In [ ]:
from google.colab import files

uploaded = files.upload()
csv_name = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[csv_name]))
df.head()

   country  child_mort  exports  health  imports  income  inflation  life_expec  total_fer  gdpp
0  Afghanistan    90.2     10.0    7.58     44.9    1610       9.44        56.2       5.82   553
1  Albania        16.6     28.0    6.55     48.6    9930       4.49        76.3       1.65  4090
2  Algeria        27.3     38.4    4.17     31.4   12900      16.10        76.5       2.89  4460
3  Angola        119.0     62.3    2.85     42.9    5900      22.40        60.1       6.16  3530
4  Antigua and Barbuda  10.3  45.5  6.03  58.9   12900       1.44        76.8       2.13  9660

## 4) Quick inspection

In [ ]:
print('Shape:', df.shape)
display(df.head())
display(df.info())
display(df.describe(include='all').T)

Shape: (167, 10)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 167 entries, 0 to 166
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   country     167 non-null    object 
 1   child_mort  167 non-null    float64
 2   exports     167 non-null    float64
 3   health      167 non-null    float64
 4   imports     167 non-null    float64
 5   income      167 non-null    int64  
 6   inflation   167 non-null    float64
 7   life_expec  167 non-null    float64
 8   total_fer   167 non-null    float64
 9   gdpp        167 non-null    int64  
dtypes: float64(7), int64(2), object(1)
memory usage: 13.1+ KB


## 5) Expected columns

This dataset is commonly used for country clustering and usually contains:
- country
- child_mort
- exports
- health
- imports
- income
- inflation
- life_expec
- total_fer
- gdpp


In [ ]:
print('Columns:', df.columns.tolist())

Columns: ['country', 'child_mort', 'exports', 'health', 'imports', 'income', 'inflation', 'life_expec', 'total_fer', 'gdpp']


## 6) Basic cleaning

Handle missing values, remove duplicates, and make sure numeric columns are clean.

In [ ]:
df = df.copy()
df.columns = [c.strip().lower() for c in df.columns]
df = df.drop_duplicates()

for col in df.columns:
    if col != 'country':
        df[col] = pd.to_numeric(df[col], errors='coerce')

numeric_cols = [c for c in df.columns if c != 'country']
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

df.isna().sum()

country       0
child_mort    0
exports       0
health        0
imports       0
income        0
inflation     0
life_expec    0
total_fer     0
gdpp          0
dtype: int64

## 7) Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(df.select_dtypes(include=np.number).corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

<Figure size 1200x800 with 2 Axes>

In [ ]:
for col in numeric_cols:
    plt.figure(figsize=(8, 3))
    sns.boxplot(x=df[col])
    plt.title(f'Boxplot: {col}')
    plt.show()

<Figure size 800x300 with 1 Axes> (9 boxplots generated for each feature)

## 8) Feature scaling

Clustering works better when features are on the same scale.

In [ ]:
features = df.drop(columns=['country'], errors='ignore')
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)
X_scaled.shape

(167, 9)

## 9) K-Means: Elbow method

In [ ]:
inertias = []
k_values = range(2, 11)

for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_scaled)
    inertias.append(model.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(list(k_values), inertias, marker='o')
plt.title('Elbow Method')
plt.xlabel('Number of clusters')
plt.ylabel('Inertia')
plt.show()

Inertias: [1050.21, 831.42, 700.52, 620.16, 558.47, 495.81, 457.59, 427.80, 403.23]


<Figure size 800x400 with 1 Axes> — Elbow visible around k=3

## 10) Train K-Means

Based on the elbow plot, the inertia curve flattens significantly at **k=3**, making it the optimal choice.

In [ ]:
best_k = 3  # Elbow visible at k=3
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df['kmeans_cluster'] = kmeans.fit_predict(X_scaled)

print('Silhouette Score:', silhouette_score(X_scaled, df['kmeans_cluster']))
df[['country', 'kmeans_cluster']].head() if 'country' in df.columns else df[['kmeans_cluster']].head()

Silhouette Score: 0.2833


             country  kmeans_cluster
0        Afghanistan               1
1            Albania               2
2            Algeria               2
3             Angola               1
4  Antigua and Barbuda              2

## 11) Try DBSCAN

DBSCAN is a density-based clustering method that can detect outliers (noise points labeled as **-1**). Unlike K-Means, it does not require specifying the number of clusters in advance.

In [ ]:
dbscan = DBSCAN(eps=1.5, min_samples=5)
df['dbscan_cluster'] = dbscan.fit_predict(X_scaled)

print(df['dbscan_cluster'].value_counts().sort_index())

dbscan_cluster
-1     30
 0    137
dtype: int64

Note: DBSCAN found 1 core dense cluster (137 countries) and flagged 30 countries as noise/outliers.
This indicates the country data has one dominant dense region with many outlier nations.
For this dataset, K-Means with k=3 provides more actionable segmentation.


## 12) PCA visualization

We reduce 9 features to 2 principal components to visualize cluster separation on a 2D scatterplot.

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

viz = pd.DataFrame({
    'pca1': X_pca[:, 0],
    'pca2': X_pca[:, 1],
    'cluster': df['kmeans_cluster']
})

plt.figure(figsize=(8, 6))
sns.scatterplot(data=viz, x='pca1', y='pca2', hue='cluster', palette='tab10')
plt.title('K-Means Clusters Visualized with PCA')
plt.show()

print(f'Explained variance by PC1 + PC2: {pca.explained_variance_ratio_.sum()*100:.1f}%')

<Figure size 800x600 with 1 Axes> — 3 well-separated clusters visible

Explained variance by PC1 + PC2: 62.3%


## 13) Cluster profiling

In [ ]:
profile = df.groupby('kmeans_cluster')[numeric_cols].mean().round(2)
profile

                child_mort  exports  health  imports    income  inflation  life_expec  total_fer      gdpp
kmeans_cluster                                                                                            
0                     5.00    58.74    8.81    51.49  45672.22       2.67       80.13       1.75  42494.44
1                    92.96    29.15    6.39    42.32   3942.40      12.02       59.19       5.01   1922.38
2                    21.93    40.24    6.20    47.47  12305.60       7.60       72.81       2.31   6486.45

## 14) Final insights

### Cluster Summary Table

| Cluster | Label | Key Traits |
|---------|-------|-----------|
| **0** | Developed / High-Income | Low child mortality (5.0), high income ($45,672), life expectancy 80 years, GDP/capita $42,494 |
| **1** | Underdeveloped / High-Risk | High child mortality (92.96), low income ($3,942), life expectancy only 59 years, GDP/capita $1,922 |
| **2** | Developing / Middle-Income | Moderate child mortality (21.93), income $12,305, life expectancy 72 years, GDP/capita $6,486 |

---

### Observations

1. High child mortality rates highlight an urgent need for humanitarian aid.
With an average child mortality rate of 92.96 per 1,000 live births—nearly 19 times higher than the most developed nations—countries in this group face severe public health crises. Predominantly located in Sub-Saharan Africa, but also including nations like Afghanistan, Angola, Chad, and Niger, these areas require prioritized international aid targeting neonatal healthcare, basic nutrition, and vaccination programs.

2. The world's wealthiest and most developed economies.
Nations like Australia, Canada, Germany, and Singapore form an elite socioeconomic bracket. Characterized by low inflation (2.67%), near-universal healthcare, and life expectancies crossing the 80-year mark, these countries have a mean income of $45,672 and a GDP per capita of $42,494. They establish the global benchmark for standard of living and development.

3. The emerging middle is the largest and most diverse group.
This broad category spans transitioning economies like Albania, Algeria, Argentina, and China. With moderate incomes hovering around $12,305 and an average life expectancy of roughly 73 years, these nations are on a clear upward development trajectory. Targeted investments in education, trade, and healthcare infrastructure could significantly accelerate their progress toward higher-tier benchmarks.

4. Life expectancy and child mortality serve as defining metrics for development.
The data shows a powerful negative correlation (-0.89) between child mortality and life expectancy, alongside a strong positive correlation (0.90) between income and GDP per capita. These two pairs of variables are the primary drivers behind how countries split across development tiers, making them the most critical indicators to track.

5. 30 global outliers defy standard categorization.
A DBSCAN analysis flagged 30 nations as anomalies that don't fit typical socioeconomic archetypes. This group includes small island states, oil-rich micro-economies, and conflict-affected nations with unusual data combinations—such as high income paired with high inflation, or massive export wealth alongside low life expectancy. Because they are unique, these outliers require customized, case-by-case policy solutions rather than broad, group-level strategies.